In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


This notebook takes the food item level SHeS 2021 data with impacts, cost and disaggregated dairy food groups and calculates the per capita intake,impacts and costs at the individual level. These data serve as the baseline dataset which acts as the comparitor for simulation scenario.  

In [ ]:
%cd /content/drive/MyDrive/mSHIFT_SHeS/code_ocean/

/content/drive/MyDrive/mSHIFT_SHeS/code_ocean


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys

In [ ]:
# Add the parent directory of the notebook to sys.path, enables module imports from notebook_code
sys.path.append(str(Path().resolve() / 'code/notebooks/notebook_code'))

In [ ]:
from data_processing import nutrient_intake, add_non_diet_variables, white_processed_meat_consumption

In [ ]:
data_path = Path("data")

In [ ]:
### Lists of indicators ####
nutrients = np.loadtxt(data_path / 'indicator_lists/nutrients.txt', dtype=str).tolist()
env_columns = np.loadtxt(data_path / 'indicator_lists/env_columns.txt', dtype=str).tolist()
error_columns = np.loadtxt(data_path / 'indicator_lists/error_columns.txt', dtype=str).tolist()
all_indicators = nutrients + env_columns

meat_food_groups = np.loadtxt(data_path / 'indicator_lists/meat_food_groups.txt', dtype=str).tolist()
all_meat_food_groups = np.loadtxt(data_path / 'indicator_lists/all_meat_food_groups.txt', dtype=str).tolist()
food_groups_dairy = np.loadtxt(data_path / 'indicator_lists/food_groups_dairy.txt', dtype=str).tolist()

In [ ]:
########## SHeS 2021 ###########

# item level diet data including both foodDB indicators and dairy food groups
diet_data = pd.read_parquet(data_path / 'diet_data.parquet')
# demographic data
dem_data =  pd.read_csv(data_path / 'shes_data_raw/shes21i_eul.csv', low_memory=False, encoding = 'IBM819')

diet_data_ids = diet_data['Cpseriala'].unique().tolist()
df_baseline = pd.DataFrame(index=diet_data_ids, columns=all_indicators)

In [ ]:
!pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 25.9 MB/s eta 0:00:00


In [ ]:
import pyreadstat

In [ ]:
########## SHeS 2024 ###########

# item level diet data including both foodDB indicators and dairy food groups
diet_data = pd.read_parquet(data_path / 'diet_data_shes_2024.parquet')
# demographic data
ind_data_path = data_path / 'shes_data_raw/shes_2024/shes_2024_eul.dta'
dem_data =  dem_data, _ = pyreadstat.read_dta(ind_data_path)

# Initialise the individual level data
diet_data_ids = diet_data['CPSerialA'].unique().tolist()
df_baseline = pd.DataFrame(index=diet_data_ids, columns=all_indicators)

In [ ]:
df_baseline = df_baseline.apply(lambda row: nutrient_intake(row,
                                                            diet_data=diet_data,
                                                            nutrients=all_indicators,
                                                            error_columns=error_columns),
                                axis=1)

In [ ]:
# Add non-diet delated individual level data on age, sex, deprivation quintile and sample
df_baseline = add_non_diet_variables(df=df_baseline, dem_data=dem_data)

In [ ]:
# Compute white processed meat intake, red and red processed meat intake and all meat intake
df_baseline['white PM intake'] = df_baseline.apply(lambda row: white_processed_meat_consumption(row, diet_data=diet_data), axis=1)
df_baseline['Total intake'] = df_baseline[meat_food_groups].sum(axis=1)
df_baseline['Total RRPM baseline'] = df_baseline['Total intake'] - df_baseline['white PM intake']
df_baseline['All meat'] = df_baseline[all_meat_food_groups].sum(axis=1)

/tmp/ipython-input-3638796802.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_baseline['white PM intake'] = df_baseline.apply(lambda row: white_processed_meat_consumption(row, diet_data=diet_data), axis=1)
/tmp/ipython-input-3638796802.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_baseline['Total intake'] = df_baseline[meat_food_groups].sum(axis=1)
/tmp/ipython-input-3638796802.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

In [ ]:
# Compute baseline total dairy intake at the participant level
df_baseline['Total dairy'] = df_baseline[food_groups_dairy].sum(axis=1)

/tmp/ipython-input-2960513520.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_baseline['Total dairy'] = df_baseline[food_groups_dairy].sum(axis=1)


In [ ]:
df_baseline.to_parquet(data_path / 'df_baseline.parquet')

In [ ]:
df_baseline = pd.read_parquet(data_path / 'df_baseline.parquet')

In [ ]:
food_group_columns = ['Fruitg',
'DriedFruitg',
'FruitJuiceg',
'SmoothieFruitg',
'Tomatoesg',
'TomatoPureeg',
'Brassicaceaeg',
'YellowRedGreeng',
'Beansg',
'Nutsg',
'OtherVegg',
'Beefg',
'Lambg',
'Porkg',
'ProcessedRedMeatg',
'OtherRedMeatg',
'Burgersg',
'Sausagesg',
'Offalg',
'Poultryg',
'ProcessedPoultryg',
'GameBirdsg',
'WhiteFishg',
'OilyFishg',
'CannedTunag',
'Shellfishg',
'Milk_Skimmed',
'Milk_SemiSkimmed',
'Milk_Whole',
'Cheese_Skimmed',
'Cheese_SemiSkimmed',
'Cheese_Whole',
'Yogurt_Skimmed',
'Yogurt_SemiSkimmed',
'Yogurt_Whole',
'Cream_SemiSkimmed',
'Cream_Whole',
'Butter'
                      ]

In [ ]:
for col in food_group_columns:
  average = np.average(df_baseline[col], weights=df_baseline['Sample Weight'])
  print(f"'{col}': {np.round(average, 2)}")
#np.average(df_baseline[food_group_columns], weights=df_baseline['Sample Weight'], axis=0)

'Fruitg': 100.66
'DriedFruitg': 3.76
'FruitJuiceg': 46.39
'SmoothieFruitg': 2.04
'Tomatoesg': 33.08
'TomatoPureeg': 4.32
'Brassicaceaeg': 17.13
'YellowRedGreeng': 21.04
'Beansg': 9.53
'Nutsg': 4.02
'OtherVegg': 48.32
'Beefg': 14.28
'Lambg': 1.89
'Porkg': 4.49
'ProcessedRedMeatg': 15.53
'OtherRedMeatg': 0.36
'Burgersg': 3.45
'Sausagesg': 7.5
'Offalg': 0.94
'Poultryg': 31.1
'ProcessedPoultryg': 0.55
'GameBirdsg': 0.31
'WhiteFishg': 6.61
'OilyFishg': 5.64
'CannedTunag': 2.42
'Shellfishg': 1.48
'Milk_Skimmed': 29.75
'Milk_SemiSkimmed': 121.02
'Milk_Whole': 32.68
'Cheese_Skimmed': 0.44
'Cheese_SemiSkimmed': 4.95
'Cheese_Whole': 16.12
'Yogurt_Skimmed': 3.23
'Yogurt_SemiSkimmed': 5.7
'Yogurt_Whole': 13.44
'Cream_SemiSkimmed': 1.44
'Cream_Whole': 2.29
'Butter': 7.96
